# Session 4 - Querying BigQuery
## From Connection to Simple Analytical SQL

This notebook is part of the *Head of Data 101* end-to-end project.

Goal of this notebook:
- Connect to BigQuery from Python
- Run basic to intermediate SELECT queries
- Inspect and visualize query results

Design principles:
- Every SQL statement is explained in plain English
- Every code line is commented for beginners
- Queries go from simple to more complex
- Focus on reading data, not modifying it

---


## Environment setup (installations)

Before running the notebook, install the required Python packages in the same environment as the notebook kernel.

Minimum required packages:
- `google-cloud-bigquery`
- `db-dtypes`
- `pandas`
- `matplotlib`

Recommended one-liner (includes the pandas extras for BigQuery):
```bash
pip install "google-cloud-bigquery[pandas]" db-dtypes pandas matplotlib
```

If you are using the `.venv` from this repo, run:
```powershell
.\.venv\Scripts\python.exe -m pip install "google-cloud-bigquery[pandas]" db-dtypes pandas matplotlib
```

---


## 1. Prerequisites and context

We will connect to a BigQuery project that already contains the analytical tables.

Project and dataset used in this course:
- Project: `albertheadofdata101`
- Dataset: `autoscout`

Tables (star schema):
- `dim_model` (make, model)
- `dim_fuel` (fuel_type)
- `dim_country` (listing_country)
- `fact_listings` (price, mileage, registration, foreign keys)

Important: BigQuery is a paid service. These queries are small, but always be mindful of costs.

---


## 2. Authentication (one-time setup)

BigQuery needs credentials to know who you are. There are two common options:

Option A: Use your Google account (recommended for local development).
1. Install the Google Cloud SDK
   - Download: https://cloud.google.com/sdk/docs/install
   - Follow the installer steps for your operating system
2. Open a terminal and run the login command:
   - `gcloud auth application-default login`
3. A browser window will open so you can sign in
4. The command stores credentials locally, and your notebook will use them automatically

Option B: Use a service account key (common for servers).
1. Create a service account and download the JSON key
2. Set the environment variable `GOOGLE_APPLICATION_CREDENTIALS` to the JSON file path

We will not store any secrets in the notebook.

---


In [ ]:
# Import the libraries we will use for BigQuery and data handling
from google.cloud import bigquery  # BigQuery client library
import pandas as pd  # Work with tabular results in Python
import matplotlib.pyplot as plt  # Create simple charts


In [ ]:
# Define the project and dataset we want to work with
project_id = 'albertheadofdata101'  # Replace if your project is different
dataset_id = 'autoscout'  # Dataset where the tables live

# Create a BigQuery client (this uses your default credentials)
client = bigquery.Client(project=project_id)

# Quick check: list the tables in the dataset
tables = client.list_tables(f'{project_id}.{dataset_id}')
[table.table_id for table in tables]  # Show table names


## 3. A helper function to run SQL and get a DataFrame

We will use a small function to avoid repeating the same boilerplate code.
It runs a SQL query and returns the results as a pandas DataFrame.

---


In [ ]:
# Helper: run a SQL query and return a DataFrame
def run_query(sql):
    # Send the SQL query to BigQuery
    query_job = client.query(sql)

    # Wait for the job to finish and download the results
    df = query_job.to_dataframe()

    # Return the DataFrame to the caller
    return df


## 4. Query 1 - List available cars (simple SELECT)

Goal: show a small sample of cars with their basic attributes.

SQL concepts introduced:
- `SELECT` to choose columns
- `FROM` to pick the table
- `JOIN` to combine fact and dimension tables
- `LIMIT` to avoid huge outputs

We use `fact_listings` for numeric data and join dimensions to get readable labels.

---


In [ ]:
# Build the SQL query as a multi-line string
sql_query_1 = f'''
SELECT
  dm.make AS make,
  dm.model AS model,
  df.fuel_type AS fuel_type,
  dc.listing_country AS listing_country,
  fl.price_eur AS price_eur,
  fl.mileage_km AS mileage_km
FROM `{project_id}.{dataset_id}.fact_listings` AS fl
JOIN `{project_id}.{dataset_id}.dim_model` AS dm
  ON fl.model_id = dm.model_id
JOIN `{project_id}.{dataset_id}.dim_fuel` AS df
  ON fl.fuel_id = df.fuel_id
JOIN `{project_id}.{dataset_id}.dim_country` AS dc
  ON fl.country_id = dc.country_id
LIMIT 20
'''

# Run the query and store the results
df_q1 = run_query(sql_query_1)

# Show the first rows
df_q1


## 5. Query 2 - Filter by country and fuel type

Goal: select cars from a specific country and engine type.

SQL concepts introduced:
- `WHERE` to filter rows
- Multiple conditions with `AND`

---


In [ ]:
# Set the filters we want to use in the query
filter_country = 'germany'  # Example country
filter_fuel = 'diesel'  # Example fuel type

# Build the SQL query using the filters above
sql_query_2 = f'''
SELECT
  dm.make AS make,
  dm.model AS model,
  df.fuel_type AS fuel_type,
  dc.listing_country AS listing_country,
  fl.price_eur AS price_eur
FROM `{project_id}.{dataset_id}.fact_listings` AS fl
JOIN `{project_id}.{dataset_id}.dim_model` AS dm
  ON fl.model_id = dm.model_id
JOIN `{project_id}.{dataset_id}.dim_fuel` AS df
  ON fl.fuel_id = df.fuel_id
JOIN `{project_id}.{dataset_id}.dim_country` AS dc
  ON fl.country_id = dc.country_id
WHERE dc.listing_country = '{filter_country}'
  AND df.fuel_type = '{filter_fuel}'
LIMIT 20
'''

# Run the query and display results
df_q2 = run_query(sql_query_2)
df_q2


## 6. Query 3 - Top 10 most expensive cars for a model

Goal: find the 10 most expensive listings for a given model.

SQL concepts introduced:
- `ORDER BY` to sort rows
- `DESC` for descending order
- `LIMIT` to keep only the top results

---


In [ ]:
# Define the model we want to analyze
target_model = 'a3'  # Example model

# Build the SQL query for the top 10 prices
sql_query_3 = f'''
SELECT
  dm.make AS make,
  dm.model AS model,
  fl.price_eur AS price_eur,
  fl.mileage_km AS mileage_km,
  df.fuel_type AS fuel_type,
  dc.listing_country AS listing_country
FROM `{project_id}.{dataset_id}.fact_listings` AS fl
JOIN `{project_id}.{dataset_id}.dim_model` AS dm
  ON fl.model_id = dm.model_id
JOIN `{project_id}.{dataset_id}.dim_fuel` AS df
  ON fl.fuel_id = df.fuel_id
JOIN `{project_id}.{dataset_id}.dim_country` AS dc
  ON fl.country_id = dc.country_id
WHERE dm.model = '{target_model}'
ORDER BY fl.price_eur DESC
LIMIT 10
'''

# Run the query and show the top 10 listings
df_q3 = run_query(sql_query_3)
df_q3


## 7. Query 4 - Count cars by model

Goal: count how many listings each model has.

SQL concepts introduced:
- `COUNT(*)` to count rows
- `GROUP BY` to aggregate by a category
- `ORDER BY` with an aggregated column

---


In [ ]:
# Build the SQL query for counting listings by model
sql_query_4 = f'''
SELECT
  dm.model AS model,
  COUNT(*) AS total_listings
FROM `{project_id}.{dataset_id}.fact_listings` AS fl
JOIN `{project_id}.{dataset_id}.dim_model` AS dm
  ON fl.model_id = dm.model_id
GROUP BY dm.model
ORDER BY total_listings DESC
'''

# Run the query and display the results
df_q4 = run_query(sql_query_4)
df_q4.head(10)  # Show the top 10 models


In [ ]:
# Plot the top 10 models by number of listings
top10_models = df_q4.head(10)

plt.figure(figsize=(10, 4))
plt.bar(top10_models['model'], top10_models['total_listings'])
plt.title('Top 10 Models by Number of Listings')
plt.xlabel('Model')
plt.ylabel('Number of Listings')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 8. Query 5 - Count cars by model and country

Goal: see how listings are distributed across models and countries.

SQL concepts introduced:
- `GROUP BY` with more than one column
- Ordering by multiple fields

---


In [ ]:
# Build the SQL query for counts by model and country
sql_query_5 = f'''
SELECT
  dm.model AS model,
  dc.listing_country AS listing_country,
  COUNT(*) AS total_listings
FROM `{project_id}.{dataset_id}.fact_listings` AS fl
JOIN `{project_id}.{dataset_id}.dim_model` AS dm
  ON fl.model_id = dm.model_id
JOIN `{project_id}.{dataset_id}.dim_country` AS dc
  ON fl.country_id = dc.country_id
GROUP BY dm.model, dc.listing_country
ORDER BY dm.model ASC, total_listings DESC
'''

# Run the query and display a sample of the results
df_q5 = run_query(sql_query_5)
df_q5.head(20)


## 9. Query 6 - Average price by model and country

Goal: compare average prices across models and countries.

SQL concepts introduced:
- `AVG()` to calculate means
- Rounding for readability

---


In [ ]:
# Build the SQL query for average price by model and country
sql_query_6 = f'''
SELECT
  dm.model AS model,
  dc.listing_country AS listing_country,
  ROUND(AVG(fl.price_eur), 2) AS avg_price_eur
FROM `{project_id}.{dataset_id}.fact_listings` AS fl
JOIN `{project_id}.{dataset_id}.dim_model` AS dm
  ON fl.model_id = dm.model_id
JOIN `{project_id}.{dataset_id}.dim_country` AS dc
  ON fl.country_id = dc.country_id
GROUP BY dm.model, dc.listing_country
ORDER BY avg_price_eur DESC
'''

# Run the query and show the results
df_q6 = run_query(sql_query_6)
df_q6.head(20)


In [ ]:
# Visualize the average price for a single model across countries
model_for_plot = 'a3'  # Pick a model you want to compare

# Filter the data for the chosen model
df_model = df_q6[df_q6['model'] == model_for_plot]

plt.figure(figsize=(8, 4))
plt.bar(df_model['listing_country'], df_model['avg_price_eur'])
plt.title(f'Average Price for {model_for_plot.upper()} by Country')
plt.xlabel('Country')
plt.ylabel('Average Price (EUR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 10. Summary

You have now: 
- Connected to BigQuery from Python
- Run multiple SELECT queries from simple to aggregated
- Visualized results with basic charts

Next steps could include:
- Adding filters for year or mileage
- Building a small dashboard
- Comparing brands or makes across countries
